# 01.7 — ADME MMP Analysis

Recreates the Matched Molecular Pair (MMP) analysis from Fang et al. (2023), §5.4/5.5, on the ADME dataset the paper released publicly (their MMP analysis itself used >25,000 confidential in-house compounds, never shared). Uses [`mmpdb`](https://github.com/rdkit/mmpdb) via `src/mmp/` (see [`src/mmp/CLAUDE.md`](../src/mmp/CLAUDE.md)) to fragment + index the dataset and extract statistically significant transformation rules per endpoint.

**Data source**: molecules and labels come from `df_sdf` (`data/processed/section4_df_sdf.pkl`), the same SDF-standardized, ChEMBL-augmented dataset built in `01.5_adme_biogen_public_recreation.ipynb` §2.4 — not the raw CSV. Two reasons: (1) the CSV and SDF standardize molecules slightly differently, which would shift fragmentation/pairing versus the paper's own approach; (2) the SDF recovers ChEMBL-augmented PPB_H/PPB_R data (1795/876 non-missing vs ~170 in the CSV), so all 6 endpoints — HLM, MDR1, SOL, RLM, PPB_H, PPB_R — are covered here, not just the 4 used for ML modelling elsewhere in this project.

**Method (matches the paper, confirmed against mmpdb source):**
- Fragmentation & indexing use mmpdb's own defaults — the paper's quoted cut-SMARTS/heavy-atom/rotatable-bond parameters are literally mmpdb's built-in defaults, not a custom rule set. Pinned explicitly in `src/mmp/mmp.py` so a future mmpdb version can't silently drift.
- "Representative rules" = per-rule statistics with ≥5 matched pairs, paired-t-test p<0.05, at the most specific environment radius ≤3 — mmpdb always computes radius 0–5 at index time; the paper's "max radius 3" is a query-time selection, reimplemented in `significant_rules()`.

**Scope**: straight recreation only (single run on the full dataset). A data-quantity ablation (subsampling to see how the significant-rule count degrades with N, mirroring the project's learning-curve experiments) and a parameter-sensitivity pass are deferred.

## 1 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import time
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt

from src.mmp import write_smi_file, write_properties_file, run_fragment, run_index, significant_rules

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
MMP_DIR = DATA_PROC / 'mmp'
MMP_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINTS = {
    'HLM':   'LOG HLM_CLint (mL/min/kg)',
    'MDR1':  'LOG MDR1-MDCK ER (B-A/A-B)',
    'SOL':   'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'RLM':   'LOG RLM_CLint (mL/min/kg)',
    'PPB_H': 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'PPB_R': 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
}

## 2 — Load data and write mmpdb input files

Loads `df_sdf` (standardized molecules, keyed by canonical SMILES, no separate compound-ID column — one is generated below). `mmpdb`'s property-file format requires short, whitespace-free property names, so we rename the endpoint columns to their short forms before writing.

In [ ]:
df = joblib.load(DATA_PROC / 'section4_df_sdf.pkl')
df = df.rename(columns={long: short for short, long in ENDPOINTS.items()})
df.insert(0, 'compound_id', [f'mol{i}' for i in range(len(df))])
print(df.shape)
df[['compound_id', 'can_smi'] + list(ENDPOINTS.keys())].head()

In [ ]:
smi_path = MMP_DIR / 'adme.smi'
props_path = MMP_DIR / 'adme_props.csv'

write_smi_file(df, smiles_col='can_smi', id_col='compound_id', path=smi_path)
write_properties_file(df, id_col='compound_id', property_cols=list(ENDPOINTS.keys()), path=props_path)

## 3 — Fragment + index

Builds the MMP SQLite database. Takes well under a minute on this dataset size.

In [ ]:
fragments_path = MMP_DIR / 'adme.fragments'
db_path = MMP_DIR / 'adme.mmpdb'

t0 = time.time()
run_fragment(smi_path, fragments_path, num_jobs=8)
print(f'fragment: {time.time() - t0:.1f}s')

t0 = time.time()
run_index(fragments_path, props_path, db_path)
print(f'index: {time.time() - t0:.1f}s')

## 4 — Database scale vs the paper

The paper built its database from >25,000 internal compounds and reported >1.2M rules. Checking how our public set compares before filtering for significance.

In [ ]:
con = sqlite3.connect(db_path)
n_compounds_indexed = con.execute('select count(*) from compound').fetchone()[0]
n_pairs = con.execute('select count(*) from pair').fetchone()[0]
n_candidate_rules = con.execute('select count(*) from rule').fetchone()[0]
con.close()

print(f'compounds indexed (>= 1 non-missing property): {n_compounds_indexed} / {len(df)}')
print(f'matched pairs: {n_pairs}')
print(f'candidate rules (pre-significance-filter): {n_candidate_rules}')
print('paper: >25,000 compounds -> >1.2M rules')

### 4.1 — Why the rule/pair counts above don't mean that many *usable* rules per endpoint

**What a "rule" and a "pair" actually are**: mmpdb first asks a purely structural question about the whole compound set — "which molecules are near-neighbours (differ by one small, localized change), and what exactly changed?" A **pair** is one such neighbour relationship (e.g. molecule #12 and molecule #47, which differ by swapping a fluorine for a hydrogen). A **rule** is the *type* of change involved (e.g. "F → H"), pooled across every pair that makes that same swap anywhere in the dataset. Neither `rule` nor `pair` (§4, above) knows or cares whether either molecule was ever tested in the HLM/MDR1/SOL/RLM/PPB_H/PPB_R assays — they're computed from structure alone.

**Where the assay endpoint comes in**: separately, each compound has (or is missing) a measured value for each of the 6 endpoints — coverage differs endpoint by endpoint, since not every compound was run through every assay. A rule like "F → H" only becomes *usable* for, say, HLM once we restrict to the subset of its pairs where **both** molecules in the pair have a non-missing HLM value. If "F → H" occurs in 40 pairs overall but only 3 of those pairs have HLM measured on both sides, then for HLM that rule effectively has just 3 data points — likely far short of the paper's 5-pair minimum, even though the same rule might clear that bar easily for an endpoint with better coverage.

Walking that funnel by hand for one endpoint (HLM) first, then generalizing to all six below.

In [ ]:
def count_rules_with_stats(con, property_id, min_pairs=0, max_p_value=None):
    """Count distinct rules with a rule_environment_statistics row (radius<=3, count>=min_pairs) for this property."""
    query = '''select count(distinct r.id)
               from rule r
               join rule_environment re on re.rule_id = r.id
               join rule_environment_statistics res on res.rule_environment_id = re.id
               where res.property_name_id = ? and re.radius <= 3 and res.count >= ?'''
    params = [property_id, min_pairs]
    if max_p_value is not None:
        query += ' and res.p_value < ?'
        params.append(max_p_value)
    return con.execute(query, params).fetchone()[0]

In [ ]:
con = sqlite3.connect(db_path)
hlm_property_id = con.execute('select id from property_name where name = ?', ('HLM',)).fetchone()[0]

n_hlm_raw = df['HLM'].notna().sum()
print(f'raw dataset: {n_hlm_raw} / {len(df)} compounds have an HLM value')

n_hlm_indexed_with_property = con.execute(
    'select count(distinct compound_id) from compound_property where property_name_id = ?',
    (hlm_property_id,),
).fetchone()[0]
print(f'of the {n_compounds_indexed} indexed compounds (those with >=1 structural match), '
      f'{n_hlm_indexed_with_property} still have an HLM value')

n_hlm_any_stat = count_rules_with_stats(con, hlm_property_id)
print(f'of the {n_candidate_rules} candidate rules, only {n_hlm_any_stat} have >=1 matched pair '
      f'where both molecules have HLM data')

n_hlm_min_pairs = count_rules_with_stats(con, hlm_property_id, min_pairs=5)
print(f"of those, only {n_hlm_min_pairs} clear the paper's >=5-matched-pair minimum")

n_hlm_significant = count_rules_with_stats(con, hlm_property_id, min_pairs=5, max_p_value=0.05)
print(f'and {n_hlm_significant} of those also pass p<0.05 -- this is the HLM row shown in §5 below')

con.close()

### 4.2 — Same funnel, all six endpoints

Reusing `count_rules_with_stats` from 4.1 for each endpoint.

In [ ]:
con = sqlite3.connect(db_path)

funnel_rows = []
for ep in ENDPOINTS:
    property_id = con.execute('select id from property_name where name = ?', (ep,)).fetchone()[0]
    n_indexed_with_property = con.execute(
        'select count(distinct compound_id) from compound_property where property_name_id = ?',
        (property_id,),
    ).fetchone()[0]
    funnel_rows.append({
        'endpoint': ep,
        'compounds with property': n_indexed_with_property,
        'rules with >=1 matched pair': count_rules_with_stats(con, property_id),
        'rules with >=5 pairs': count_rules_with_stats(con, property_id, min_pairs=5),
        '+ p<0.05': count_rules_with_stats(con, property_id, min_pairs=5, max_p_value=0.05),
    })

con.close()

funnel_df = pd.DataFrame(funnel_rows).set_index('endpoint')
funnel_df

## 5 — Representative rules per endpoint

Filter: ≥5 matched pairs, paired-t-test p<0.05, most specific environment radius ≤3 — exactly the paper's stated criteria (§5.4).

In [ ]:
rules_by_endpoint = {}
for ep in ENDPOINTS:
    rules = significant_rules(db_path, property_name=ep, max_radius=3, min_pairs=5, max_p_value=0.05)
    rules_by_endpoint[ep] = rules
    print(f'{ep}: {len(rules)} significant rules')

In [ ]:
for ep, rules in rules_by_endpoint.items():
    print(f'--- {ep} ---')
    display(rules[['from_smiles', 'to_smiles', 'n_pairs', 'mean_change', 'std_change', 'p_value']])

## 6 — Summary: significant rules per endpoint

In [ ]:
counts = pd.Series({ep: len(rules) for ep, rules in rules_by_endpoint.items()})

fig, ax = plt.subplots(figsize=(5, 3.5))
counts.plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('significant rules (>=5 pairs, p<0.05, radius<=3)')
ax.set_xlabel('endpoint')
ax.set_title('MMP rules surviving significance filter, per endpoint')
plt.tight_layout()
plt.savefig('../figures/section5_mmp_significant_rules_per_endpoint.png', dpi=150)
plt.show()

## 7 — Discussion

The mmpdb pipeline itself recreates cleanly — fragmentation and indexing use mmpdb's own defaults (confirmed to match the paper's stated parameters verbatim by reading the mmpdb source), and the significance filter (≥5 pairs, p<0.05, radius≤3) is a direct SQL reimplementation of what mmpdb computes internally.

Using `df_sdf` instead of the raw CSV (§2) matters here in a way it didn't for the ML modelling notebooks: it's not just about matching the paper's standardization, it recovers the ChEMBL-augmented PPB_H/PPB_R labels that the public CSV drops almost entirely. That shows up directly in §4.2 — PPB_H, with by far the best property coverage among the six endpoints once augmented (1101 indexed compounds with a value vs 1584 for HLM despite HLM having ~3x more raw label coverage — PPB_H's labels land on compounds with more structural neighbors), is the only endpoint where the recreation looks like the paper's: 92 rules clear the ≥5-pair bar and **21 survive p<0.05**, with clearly interpretable SAR (F→Cl on an aniline ring drops PPB_H by ~0.74 log units; Cl→H and CF3→H both raise it; halogen and alkyl-chain-length trends are consistent in direction and magnitude across several related rules). Had we stayed on the CSV, PPB_H/PPB_R would have been unusable — the paper's ≥5-pair filter would have zeroed out almost everything, same as MDR1/SOL are now.

HLM, MDR1, SOL, and RLM tell a different story: even with the richer SDF-standardized compound set (5252 unique molecules vs the CSV's 3521, since PPB augmentation added many ChEMBL structures), only 5 candidate rules per endpoint ever reach the ≥5-pair bar (versus 92 for PPB_H), and just 1 (HLM, RLM) or 0 (MDR1, SOL) of those survive significance. This isn't a data-loading artifact — §4.1/§4.2 show the funnel explicitly: thousands of rules have *some* property data (4000+ for HLM/RLM), but almost none reach the pair-count minimum. The paper's own database used >25,000 compounds and reported >1.2M rules; ours produces ~11,900 candidate rules from 2871 indexed compounds, and PPB_H's outcome shows the method works fine here — it just needs enough compounds sharing both a structural neighbor and a measured label, and for four of the six endpoints in this public set, that combination is rare.

A natural follow-up — deferred for now — is to subsample this dataset the same way the learning-curve experiments do, and track how the significant-rule count per endpoint degrades with N, using PPB_H (which currently has real signal) as the one endpoint where a shrinking-N trend would actually be visible.